In [1]:
import pandas as pd
from typing import Tuple
import math
import numpy as np
from tqdm import tqdm

# Cleaning Schedule
This notebook provides a clean version of the raw `2025schedule.csv` file.
If does the following:
1. Rename duplicate columns.
2. Adding a 'timestamp' column, which provides a `pd.Timestamp` for the game start.
3. Adding 'homedistancetraveled' and 'visdistancetraveled' columns, which contains the distance in miles traveled from the team's previous games.
4. Adding 'homerestdays' and 'visrestdays' columns, which contains the number of days between the current game and the previous game for the home and visiting teams.
5. Save to a new csv, `.data/gameinfo_clean.csv`

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
schedule = pd.read_csv('2025schedule.csv')
schedule.head()

,Date,Num,Day,Visitor,League,Game,Home,League.1,Game.1,Day/Night,Location,Postponed,Makeup
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN


## 1. Rename Duplicate Columns

In [4]:
schedule = schedule.rename(columns={'League':'VisitorLeague', 'Game':'VisitorGame', 'League.1':'HomeLeague', 'Game.1':'HomeGame'})
schedule.head()

,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN


## 2. Adding a 'timestamp' column, which provides a `pd.Timestamp` for the game start.

In [7]:
def get_timestamp(date: int) -> pd.Timestamp:
    """For a given date int of format YYYYMMDD, gets its game start timestamp."""

    date = str(date)

    y = int(date[:4])
    m = int(date[4:6])
    d = int(date[6:])
    return pd.Timestamp(year=y, month=m, day=d)

# Add the timestamp column

schedule['timestamp'] = schedule['Date'].apply(get_timestamp)
schedule.head()

,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN,2025-03-18
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN,2025-03-19
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN,2025-03-27
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN,2025-03-27
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN,2025-03-27


## 3. Adding 'homedistancetraveled' and 'visdistancetraveled' Columns

In [8]:
# Add temporary latitude and longitude columns
parks = pd.read_csv('Parks.csv')
schedule = pd.merge(schedule, parks[['PARKID', 'Latitude', 'Longitude']], how='left', left_on='Location', right_on='PARKID').reset_index(drop=True)
schedule = schedule.drop('PARKID', axis=1)

In [9]:
def haversine(lat1, lon1, lat2, lon2):
    """Returns Haversine distance between two pairs of latitudes and longitudes."""
    R = 3958.8  # Earth radius in miles
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [10]:
def get_closest_home_game_coords(team: str, idx: int) -> Tuple[float, float]:
    """For the given team, returns the latitude and longitude of the closest home game they played
    to wherever they are in schedule, given by idx. If no home game is found, it returns None, None."""
    
    n_games = len(schedule)
    
    before = idx - 1
    after = idx + 1
    while before >= 0 or after <= n_games - 1:
        if before >= 0:
            before_gm = schedule.iloc[before]
            if before_gm['Home'] == team:
                return before_gm['Latitude'], before_gm['Longitude']
            
            before -= 1

        if after <= n_games - 1:
            after_gm = schedule.iloc[after]
            if after_gm['Home'] == team:
                return after_gm['Latitude'], after_gm['Longitude']
            
            after += 1
            
    return None, None

In [11]:
home_dists = []
away_dists = []

last_games = {} # Mapping of team id to (lat, lon, season) tuple of previous game - if empty, just use np.nan

for i, game in tqdm(schedule.iterrows()):
    
    cur_lat = game['Latitude']
    cur_lon = game['Longitude']
    
    home_team = game['Home']
    away_team = game['Visitor']
    
    for team, dists in zip([home_team, away_team], [home_dists, away_dists]):
        if team in last_games:
            last_lat, last_lon = last_games[team]           
            
        else: # Never played a game before
            last_lat, last_lon = get_closest_home_game_coords(team, i)
            
        if last_lat is not None:
            dist = haversine(last_lat, last_lon, cur_lat, cur_lon)
            dists.append(dist)
        else:
            dists.append(np.nan)
            
        last_games[team] = (cur_lat, cur_lon)

schedule['homedistancetraveled'] = home_dists    
schedule['visdistancetraveled'] = away_dists
schedule.head()

2430it [00:00, 25825.49it/s]


,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp,Latitude,Longitude,homedistancetraveled,visdistancetraveled
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN,2025-03-18,35.705526,139.751928,0.0,5473.620854
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN,2025-03-19,35.705526,139.751928,0.0,0.000000
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN,2025-03-27,40.829586,-73.926413,0.0,736.818918
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN,2025-03-27,43.641256,-79.389054,0.0,333.374579
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN,2025-03-27,32.747361,-97.084167,0.0,1562.150215


## 4. Add Rest Day Columns

In [12]:
home_rest_days = []
away_rest_days = []

last_played = {}

for _, game in tqdm(schedule.iterrows()):
    home_team = game['Home']
    away_team = game['Visitor']
    timestamp = game['timestamp']
    
    prev_home_t = last_played.get(home_team)
    prev_away_t = last_played.get(away_team)
    
    home_rest_days.append((timestamp.floor('D') - prev_home_t.floor('D')).days if prev_home_t is not None else np.nan)
    away_rest_days.append((timestamp.floor('D') - prev_away_t.floor('D')).days if prev_away_t is not None else np.nan)

    last_played[home_team] = timestamp
    last_played[away_team] = timestamp
 
schedule['homerestdays'] = home_rest_days
schedule['visrestdays'] = away_rest_days
schedule.head()

2430it [00:00, 7691.95it/s]


,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp,Latitude,Longitude,homedistancetraveled,visdistancetraveled,homerestdays,visrestdays
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN,2025-03-18,35.705526,139.751928,0.0,5473.620854,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN,2025-03-19,35.705526,139.751928,0.0,0.000000,1.0,1.0
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN,2025-03-27,40.829586,-73.926413,0.0,736.818918,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN,2025-03-27,43.641256,-79.389054,0.0,333.374579,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN,2025-03-27,32.747361,-97.084167,0.0,1562.150215,NaN,NaN


## 5. Save to `.csv`

In [13]:
schedule.to_csv('./2025schedule_cleaned.csv', index=False)